# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower, regexp_replace

# Read from bronze table

In [0]:
df = spark.table("bike_data.bronze.cust_az12_raw")

# Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

## Normalization

In [0]:
# change the string values to be small letters
df = (df
    .withColumn("GEN", lower(col("GEN")))
    .withColumn("CID", lower(col("CID")))
)
df.display()

In [0]:
df =( df
    .withColumn(
        "GEN",
        F.when(col("GEN") == "m", "male")
        .when(col("GEN") == "f", "female")
        .otherwise(col("GEN"))
    )     
)

### Customer Key Normalization

In [0]:
# Remove NAS prefix from ERP Customer Identifier
df = (df
      .withColumn("CID", regexp_replace(col("CID"), "^nas", ""))
      )
df.display()

## Renaming Columns

In [0]:
# make the ids use lowercase
df = (df
      .withColumnRenamed("CID", "customer_key")
      .withColumnRenamed("BDATE", "dob")
      .withColumnRenamed("GEN", "gender")
)
df.display()

# Write to silver

In [0]:
(
  df.write
  .mode("overwrite")
  .saveAsTable("bike_data.silver.erp_customer_demographics")
)